In [ ]:
from pathlib import Path

import pandas as pd

rootPath = Path().cwd().parent.parent

power_plants_data_path = rootPath.joinpath("data", "power_plants", "global_power_plant_database.csv")
power_plants_data = pd.read_csv(power_plants_data_path, low_memory=False)
power_plants_data.head()


In [ ]:
AFRICAN_COUNTRIES = (
    "Algeria",
    "Angola",
    "Benin",
    "Botswana",
    "Burkina Faso",
    "Burundi",
    "Cameroon",
    "Cape Verde",
    "Central African Republic",
    "Congo",
    "Cote DIvoire",
    "Democratic Republic of the Congo",
    "Djibouti",
    "Egypt",
    "Equatorial Guinea",
    "Eritrea",
    "Ethiopia",
    "Gabon",
    "Gambia",
    "Ghana",
    "Guinea",
    "Guinea-Bissau",
    "Kenya",
    "Lesotho",
    "Liberia",
    "Libya",
    "Madagascar",
    "Malawi",
    "Mali",
    "Mauritania",
    "Mauritius",
    "Morocco",
    "Mozambique",
    "Namibia",
    "Niger",
    "Nigeria",
    "Rwanda",
    "Senegal",
    "Sierra Leone",
    "South Africa",
    "Sudan",
    "Swaziland",
    "Tanzania",
    "Togo",
    "Tunisia",
    "Uganda",
    "Western Sahara",
    "Zambia",
    "Zimbabwe",
)

In [ ]:
african_power_plants = power_plants_data.query(
    "country_long in @AFRICAN_COUNTRIES"
)
african_power_plants.to_csv(power_plants_data_path.parent / "african_power_plants.csv", index=False)

african_power_plants.head()

In [ ]:
power_plants = african_power_plants

In [ ]:
import matplotlib.pyplot as plt

# Plot longitude on X, latitude on Y
plt.figure(figsize=(10, 8))
plt.scatter(power_plants['longitude'], power_plants['latitude'])
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Power Plants in the Region')
plt.show()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Create Point objects from lat/lon
geometry = [Point(lon, lat) for lon, lat in
            zip(power_plants['longitude'], power_plants['latitude'])]

# Convert to GeoDataFrame
power_plants_gdf = gpd.GeoDataFrame(
    power_plants,
    geometry=geometry,
    crs="EPSG:4326"  # This tells GeoPandas we're using standard lat/lon
)

power_plants_gdf.head()

In [ ]:
power_plants_gdf.plot(figsize=(10, 8), marker='o', color='red', markersize=50)
plt.title('Power Plants in the Region')
plt.show()

In [ ]:
countries = gpd.read_file(rootPath / "data" / "geo" / "africa" / "africa.geo.json")
countries.plot(figsize=(10, 8), color='lightgray', edgecolor='black')
countries.head()


In [ ]:
power_plants_tagged = gpd.sjoin(
    power_plants_gdf,      # Left: the data we want to tag
    countries,         # Right: the polygons with labels we want
    how="left",        # Keep all power plants, even if no match
    predicate="within" # The spatial relationship to test
)

In [ ]:
print(countries.shape)
print(power_plants_gdf.shape)
print(power_plants_tagged.shape)


In [ ]:
power_plants_tagged.head(10)

In [ ]:
fuel_by_country = power_plants_tagged.groupby(['name_long', 'primary_fuel']).size().unstack(fill_value=0)
fuel_by_country.loc[['Zambia', 'Malawi']]

In [ ]:
import contextily as cx
# Create a figure
fig, ax = plt.subplots(figsize=(12, 10))

# Plot our data (must be in Web Mercator projection for contextily)
power_plants_web = power_plants_gdf.to_crs(epsg=3857)  # Convert to Web Mercator
power_plants_web.plot(ax=ax, marker='o', color='red', markersize=50, zorder=5)

# Add african boundaries
countries_web = countries.to_crs(epsg=3857)

# --- Three-layer example with transparency ---
# Layer 1 - base fill with transparency (alpha)
# countries_web.plot(ax=ax, color='lightgray', edgecolor='none', alpha=0.4, zorder=1)

# Layer 2 - optional highlight for a couple of countries (example: Zambia & Malawi)
# Note: the GeoDataFrame is expected to have a 'name_long' column (used elsewhere in the notebook)

highlight_countries = countries_web[countries_web.get('name_long', pd.Series()).isin(['Zambia', 'Malawi'])]
highlight_countries.plot(ax=ax, column='name_long', edgecolor='orange', alpha=0.6, zorder=4)

# Layer 3 - country boundaries on top with higher opacity
# countries_web.boundary.plot(ax=ax, linewidth=0.6, edgecolor='black', alpha=0.8, zorder=3)

# Add the basemap AFTER plotting data
cx.add_basemap(ax, source=cx.providers.OpenTopoMap)

ax.set_title('Power Plants in Africa ', fontsize=14)
ax.set_axis_off()  # Hide the axis numbers
plt.show()

In [ ]:
cx.providers

In [ ]:
# Compare two styles side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Style 1: OpenTopoMap (terrain)
power_plants_web.plot(ax=axes[0], color='red', markersize=30)
cx.add_basemap(axes[0], source=cx.providers.OpenTopoMap)
axes[0].set_title('OpenTopoMap Style')
axes[0].set_axis_off()

# Style 2: CartoDB Positron (clean for reports)
power_plants_web.plot(ax=axes[1], color='red', markersize=30)
cx.add_basemap(axes[1], source=cx.providers.CartoDB.Positron)
axes[1].set_title('CartoDB Positron Style')
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Load country boundaries (same as Lessons 7.1 and 7.2)

fig, ax = plt.subplots(figsize=(14, 12))

# Project all data to Web Mercator
countries_web = countries.to_crs(epsg=3857)
power_plants_web = power_plants_gdf.to_crs(epsg=3857)

# LAYER 2 (middle): Country boundaries
countries_web.plot(
    ax=ax,
    facecolor='none',        # Transparent fill
    edgecolor='darkblue',    # Blue boundary lines
    linewidth=1.5,
    zorder=2                 # Drawing order
)

# LAYER 3 (top): Power plants colored by fuel type
power_plants_web.plot(
    ax=ax,
    column='primary_fuel',   # Color by fuel type
    cmap='Set1',
    markersize=80,
    legend=True,
    legend_kwds={'title': 'Fuel Type', 'loc': 'lower right'},
    zorder=3                 # On top of boundaries
)

# LAYER 1 (bottom): Basemap — added last but drawn at the bottom
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=1)

ax.set_title('Power Plants by Fuel Type — Africa', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Load country boundaries (same as Lessons 7.1 and 7.2)

fig, ax = plt.subplots(figsize=(14, 12))

# Project all data to Web Mercator
countries_web = countries.to_crs(epsg=3857)
power_plants_web = power_plants_gdf.to_crs(epsg=3857)

# LAYER 1 (top): Power plants coloured by fuel type
power_plants_web.plot(
    ax=ax,
    column='primary_fuel',   # Color by fuel type
    cmap='Set1',
    markersize=80,
    legend=True,
    legend_kwds={'title': 'Fuel Type', 'loc': 'lower right'},
    zorder=3
)

# LAYER 2 (bottom): Country boundaries
countries_web.plot(
    ax=ax,
    facecolor='none',        # Transparent fill
    edgecolor='darkblue',    # Blue boundary lines
    linewidth=1.5,
    zorder=2
)

# LAYER 3 (basemap): Added last — contextily draws it behind everything
cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)

ax.set_title('Power Plants by Fuel Type — Africa', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()